In [9]:
using Revise
using InteractiveUtils

const PATH_AUGMENT_QOG_JL = "phase0/functions/qog_augmented_standard.jl"
const PATH_PDF_EXTRACT_JL = "phase0/functions/qog_pdf_extract.jl"
const PATH_METADATA_ENHANCE_JL = "phase0/functions/qog_metadata_join.jl"
const PATH_CLUSTERING_JL = "phase0/functions/cluster_analysis.jl"
const PATH_METADATA_ENRICH_JL = "phase0/functions/enrich_metadata.jl"

includet(PATH_AUGMENT_QOG_JL)
includet(PATH_METADATA_ENRICH_JL)
includet(PATH_METADATA_ENHANCE_JL)
includet(PATH_CLUSTERING_JL)

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [10]:
# summarize_constants_from_file(PATH_AUGMENT_QOG_JL)

In [11]:
# document_functions_precise(PATH_AUGMENT_QOG_JL)

In [12]:
# summarize_constants_from_file(PATH_METADATA_ENHANCE_JL)

In [13]:
# document_functions_precise(PATH_METADATA_ENHANCE_JL)

In [14]:
# summarize_constants_from_file(PATH_PDF_EXTRACT_JL)

In [15]:
# document_functions_precise(PATH_CLUSTERING_JL)

In [16]:
run_enrich_metadata_samples();


  enrich_metadata.jl — Function intent and usage

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the joined metadata in one call.
│  USE WHEN: You need both df and meta_df for auditing or enrichment.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_JOINED
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ classify_temporal_profile(birth_year, death_year; kwargs...)
│  INTENT: Classify a variable's temporal profile from its lifespan.
│  USE WHEN: You have first/last year and want :anchor, :experimental,
│           :legacy, :historical, :current, :modern, or :unclassified.
│
│  ARGUMENTS:
│    birth_year::Int, death_year::Int (positional)
│    Optional kwargs: data_start, data_end, current_year, active_lag, thresholds
│
│  RETURNS: Symbol (e.g. :anchor, :current)
│
│  USAGE:
│    profile = classify_temporal_profile(1946, 2022)


In [17]:

# From metadata
show_usage();

═══════════════════════════════════════════════════════════════════════════
QoG METADATA JOINING - PHASE 0
═══════════════════════════════════════════════════════════════════════════

QUICK START
-----------

```julia
# Load the module
include("functions/qog_metadata_join.jl")

# Run complete pipeline (single isomorphism check; fails if full PDF not isomorphic)
metadata = join_metadata()

# Or: run isomorphism cascade (strictest → loosest until success), then union on slug
metadata = join_metadata_with_cascade()

# Inspect results
first(metadata, 10)
```

DIAGNOSTIC TOOLS
----------------

```julia
# Quick file check (without processing)
quick_check()

# Review exception configurations
inspect_exceptions()
```

STEP-BY-STEP EXECUTION
----------------------

```julia
# Step 1: Ingest and normalize (lowercase, ligature→ASCII, SLUG_CORRECTIONS; PDF_ONLY_PREFIXES removed; returns 4th: pdf_prefixes_df)
(stata_df, pdf_df, arrow_df, pdf_prefixes_df) = ingest_and_normalize()

# Step 2: Align I

In [2]:
# run_enrichment_examples()

In [9]:
run_augmented_standard_samples()

QoG AUGMENTED STANDARD - Complete Pipeline Guide

📖 This is DOCUMENTATION ONLY — no code is executed.
   Copy code blocks to your REPL or notebook to run them.

┌─────────────────────────────────────────────────────────────────────────┐
│ PREREQUISITES                                                           │
└─────────────────────────────────────────────────────────────────────────┘

Before running the augmentation pipeline, ensure you have:

1. Julia packages installed
2. QoG source files downloaded (or use `download_qog_sources()`)
3. Working directory set to project root

```julia
# Load dependencies
using Arrow, DataFrames, CSV, Statistics, StatsBase
using ReadStatTables, HTTP, JSON, Downloads

# Load the augmentation functions
include("functions/qog_augmented_standard.jl")

# Verify working directory
println("Working directory: ", pwd())
println("Data directory exists: ", isdir(PATH_DATA_DIR))
```

Key Path Constants:

| Constant | Description |
|----------|-------------|
| `PA